# Configuring the MISMIP ISSM model

This notebook integrates the logic from [`runme.py`](./runme.py) and [`Mismip.py`](./Mismip.py) into one readable workflow.

The main idea is:

- `runme.py` controls the experiment workflow: paths, model choice, mesh resolution, cluster target, pipeline stages, flow-equation branches, and solves.
- `Mismip.py` defines the physical model setup: geometry, friction, rheology, boundary conditions, forcing, constants, and initial conditions.

This notebook is written as a guide to the logical path. Some cells are illustrative and safe to read without launching an ISSM solve. The solve cells should only be run once the ISSM environment, Gadi paths, and previous model stages are set correctly.

## Workflow map

The model setup follows this sequence:

1. Choose the experiment family: resolution and basal friction style.
2. Choose the compute target: Gadi or local/generic.
3. Create an ISSM `organizer` to manage restartable stages.
4. Generate a mesh from `Domain.exp`.
5. Parameterize the mesh using `Mismip.py`.
6. Spin up the model with transient SSA relaxation.
7. Optionally continue the spin-up with restart stages.
8. Optionally extrude the model to 3D.
9. Run solver-comparison branches: SSA, MOLHO, HO, FS.
10. Optionally run enhanced-rheology or ESTAR branches.

A useful mental model: first create the physical experiment, then choose which stress-balance approximation to test.

## 1. Choose the experiment family

`runme.py` uses `modelnum` to choose both mesh resolution and basal friction type.

The current script sets:

```python
steps = [11]
modelnum = 1
```

That means `1km_viscous`, and only one selected organizer stage is intended to run.

In [ ]:
# Notebook representation of the model choices in runme.py
MODEL_CHOICES = {
    1: {"name": "1km_viscous", "hmax": 1000, "friction": "viscous"},
    2: {"name": "2km_viscous", "hmax": 2000, "friction": "viscous"},
    3: {"name": "1km_coulomb", "hmax": 1000, "friction": "coulomb"},
    4: {"name": "2km_coulomb", "hmax": 2000, "friction": "coulomb"},
    5: {"name": "500m_viscous", "hmax": 500, "friction": "viscous"},
    6: {"name": "500m_coulomb", "hmax": 500, "friction": "coulomb"},
    7: {"name": "200m_viscous", "hmax": 200, "friction": "viscous"},
    8: {"name": "200m_coulomb", "hmax": 200, "friction": "coulomb"},
}

modelnum = 1
choice = MODEL_CHOICES[modelnum]
modelname = choice["name"]
hmax = choice["hmax"]
friction_style = choice["friction"]

choice

### Choice consequences

This one number affects multiple later decisions:

- `hmax` controls mesh spacing in `bamg`.
- viscous cases keep the default friction created by `Mismip.py`.
- Coulomb cases replace the friction object during solve setup.
- the model name becomes part of the output directory and saved model prefix.

So changing `modelnum` is the simplest way to switch experiment families.

## 2. Prepare ISSM imports

The top of `runme.py` requires `ISSM_DIR` and then manually extends `sys.path` so ISSM Python modules can be imported.

This pattern is common in ISSM Python workflows because many modules live inside the ISSM source tree rather than in a normal Python package installed into the environment.

In [ ]:
# This mirrors the setup in runme.py. Run this only in an environment where ISSM_DIR is set.
import os
import sys

issm_dir = os.getenv("ISSM_DIR")
if issm_dir is None:
    raise RuntimeError("ISSM_DIR is not set. Load/configure ISSM before running ISSM cells.")

sys.path.insert(0, os.path.join(issm_dir, "src", "m", "boundaryconditions"))
sys.path.insert(0, os.path.join(issm_dir, "src", "m", "classes"))
sys.path.insert(1, os.path.join(issm_dir, "src", "m", "classes", "clusters"))
sys.path.append(os.path.join(issm_dir, "src", "m", "contrib", "jkjhew"))
sys.path.append(os.path.join(issm_dir, "bin"))
sys.path.append(os.path.join(issm_dir, "lib"))

issm_dir

## 3. Choose where the model runs

`runme.py` currently assumes `clustername = 'gadi'` and fills in a Gadi cluster object with hard-coded account paths.

This is one of the most user-specific parts of the script. Before running solves, check:

- the login username
- the NCI project code
- the ISSM source path
- the ISSM binary path
- the execution path on scratch
- the requested nodes, CPUs, and walltime

If those paths belong to another user, the notebook should be edited before launching jobs.

In [ ]:
# Cluster choice from runme.py, shown as configuration rather than executed by default.
clustername = "gadi"

GADI_CONFIG = {
    "name": "gadi.nci.org.au",
    "login": "jh7060",
    "srcpath": "/g/data/au88/jh7060/ISSM",
    "project": "au88",
    "numnodes": 1,
    "cpuspernode": 32,
    "time_minutes": 48 * 60,
    "codepath": "/g/data/au88/jh7060/spack/0.22/release/linux-rocky8-x86_64_v4/gcc-13.2.0/issm-4.24-v52nx3pfx7lpqwfldqlpspflk34wz756/bin",
    "executionpath": "/scratch/au88/jh7060/issm_runs",
}

GADI_CONFIG

## 4. Create the organizer

The organizer makes the workflow restartable. Instead of running every block every time, each named stage is guarded by:

```python
if org.perform('StageName'):
    ...
```

The saved model directory is based on the model choice. For `modelnum = 1`, it is:

```text
"./Models_1km_viscous"
```

The saved model prefix is:

```text
"mismip_1km_viscous_"
```

In [ ]:
# The stage list in runme.py is currently [11].
# To rebuild from scratch, you would usually run earlier stages first.
steps = [11]

organizer_config = {
    "repository": "./Models_" + modelname,
    "prefix": "mismip_" + modelname + "_",
    "steps": steps,
    "trunkprefix": "34;47;2",
}

organizer_config

## 5. Stage dependency map

The useful way to read `runme.py` is as a dependency graph.

| Stage | Depends on | Purpose |
| --- | --- | --- |
| `Mesh_generation` | `Domain.exp` | Create the 2D finite-element mesh. |
| `Parameterization` | `Mesh_generation`, `Mismip.py` | Add geometry, materials, masks, forcing, and initial conditions. |
| `Transient_Steadystate` | `Parameterization` | Long SSA transient relaxation. |
| `Transient_steadystate2/3/4/5` | Previous transient stage | Continue relaxation from the last saved solution. |
| `Transient_extrude` | `Transient_steadystate3` | Extrude to a 3D mesh and switch to HO. |
| `GlenSSA` | Spin-up or extrusion | Run SSA comparison branch. |
| `GlenMOLHO` | Spin-up or extrusion | Run MOLHO comparison branch. |
| `GlenHO` | Spin-up or extrusion | Run HO comparison branch. |
| `GlenFS` | Spin-up | Run short full-Stokes-style branch. |
| Enhanced/ESTAR branches | Spin-up or extrusion | Repeat comparison with modified material law. |

With `steps = [11]`, the script appears to target `GlenMOLHO`, assuming upstream saved models already exist.

## 6. Mesh generation

The mesh is built from [`Domain.exp`](./Domain.exp) using `bamg`.

The key choice is `hmax`, which came from `modelnum`:

- `1000` for 1 km models
- `2000` for 2 km models
- `500` for 500 m models
- `200` for 200 m models

This creates an ISSM model object `md` with mesh coordinates, elements, and mesh metadata.

In [ ]:
# Executable only after ISSM imports are available.
# from model import *
# from bamg import bamg
#
# md = bamg(model(), "domain", "./Domain.exp", "hmax", hmax, "splitcorners", 1)
# md.miscellaneous.name = "MISMIP_" + modelname
# org.savemodel(md)

## 7. Parameterization: where `Mismip.py` enters

The next stage loads the mesh and calls:

```python
md = setmask(md, '', '')
md = parameterize(md, './Mismip.py')
```

`parameterize` executes `Mismip.py` in a context where `md` already exists. That is why `Mismip.py` can directly modify `md.geometry`, `md.friction`, `md.materials`, and other fields without defining `md` itself.

## 8. Geometry in `Mismip.py`

The geometry is analytic and mesh-dependent. It uses the mesh coordinates `md.mesh.x` and `md.mesh.y`.

The script computes:

- `bx`: a polynomial bed shape in the flow direction `x`
- `by`: a transverse shape in `y`
- `by0`: the transverse term evaluated at `y = 0`
- `bed`: clipped with a lower bound of `-720`
- `surface`: clipped with a lower bound of `10`
- `base`: the maximum of bed and `-90`
- `thickness`: `surface - base`

This is the first point where the abstract mesh becomes an ice geometry.

In [ ]:
# Geometry logic from Mismip.py, shown as a compact reference.
# Requires an ISSM model object named md.
#
# import numpy as np
# bx = -150 - 728.8 * (md.mesh.x / 300000) ** 2 + 343.91 * (md.mesh.x / 300000) ** 4 - 50.57 * (md.mesh.x / 300000) ** 6
# by = (
#     500.0 / (1 + np.exp(-2.0 / 4000.0 * (md.mesh.y - 80000.0 / 2 - 24000)))
#     + 500.0 / (1 + np.exp(2.0 / 4000.0 * (md.mesh.y - 80000.0 / 2 + 24000)))
# )
# by0 = (
#     500.0 / (1 + np.exp(-2.0 / 4000.0 * (0 - 80000.0 / 2 - 24000)))
#     + 500.0 / (1 + np.exp(2.0 / 4000.0 * (0 - 80000.0 / 2 + 24000)))
# )
# md.geometry.bed = np.maximum(bx + by, -720)
# md.geometry.surface = np.maximum(bx + by0 + 100, 10)
# md.geometry.base = np.maximum(md.geometry.bed, -90)
# md.geometry.thickness = md.geometry.surface - md.geometry.base

## 9. Friction and rheology

`Mismip.py` first sets a default friction law:

- coefficient: `sqrt(3.160e6)` at every vertex
- `p = 3` on every element
- `q = 0` on every element

It also sets Glen-style material parameters:

- `rheology_B = 1 / ((6.338e-25) ** (1/3))`
- `rheology_n = 3`
- `rheology_law = 'None'`

Later, if the chosen model family is Coulomb, `runme.py` replaces the friction object before solving.

In [ ]:
# Coulomb override from runme.py, used only for modelnum 3, 4, or 6.
# Requires ISSM's frictioncoulomb class and an existing md.
#
# if friction_style == "coulomb":
#     md.friction = frictioncoulomb()
#     md.friction.coefficient = np.sqrt(3.160e6) * np.ones(md.mesh.numberofvertices)
#     md.friction.coefficientcoulomb = np.sqrt(0.5) * np.ones(md.mesh.numberofvertices)
#     md.friction.p = 3 * np.ones(md.mesh.numberofelements)
#     md.friction.q = np.zeros(md.mesh.numberofelements)

## 10. Boundary conditions and masks

`Mismip.py` applies ice-shelf boundary conditions with:

```python
md = SetIceShelfBC(md, './Front.exp')
```

Note that `Front.exp` is referenced here, but the top-level file listing currently shows `Domain.exp` and not `Front.exp`. That may be supplied elsewhere by ISSM, generated later, or missing from this checkout.

The mask logic then sets:

- `ice_levelset[:] = -1`
- `ocean_levelset[:] = -1`
- nodes near `x = 640000` are marked as an ice front with `ice_levelset = 0`

Velocity boundary conditions are also applied:

- `spcvy = 0` on the top and bottom lateral boundaries, near `y = 0` and `y = 80000`
- `spcvx = 0` and `spcvy = 0` near `x = 0`

In plain language: the script fixes motion at the upstream boundary and prevents cross-flow motion at the lateral sides.

## 11. Forcing, constants, and initial conditions

`Mismip.py` then defines the environmental and numerical setup:

Forcing:

- custom `mismipbasalforcings()` object
- melt-rate factor set to `0`
- threshold thickness `75`
- upper melt depth `-100`
- SMB mass balance `0.3` everywhere
- geothermal flux `0.5` everywhere
- grounded ice melting rate `0` everywhere

Constants and switches:

- ice density `918`
- water density `1028`
- gravity `9.8`
- seconds per year `31556926`
- thermal transient off
- grounding-line migration on with `SubelementMigration`

Initialization:

- velocity components initialized to `1`
- speed initialized to `sqrt(2)`
- pressure initialized from ice overburden
- temperature initialized to `273 K`

## 12. First solve: transient SSA spin-up

After parameterization, `runme.py` starts with an SSA transient solve:

```python
md = setflowequation(md, 'SSA', 'all')
md.timestepping.time_step = 1
md.timestepping.final_time = 200000
```

This is a long relaxation run. Its purpose is to move from the analytic initial geometry and velocity guesses toward a dynamically adjusted state.

The important choices are:

- flow equation: `SSA`
- time step: 1 year
- final time: 200,000 years
- output/checkpoint frequency: every 2,000 years
- stress-balance max iterations: 30
- relative tolerance: 1

The output is saved both through the organizer and, for the first viscous case, exported to NetCDF.

In [ ]:
# Skeleton for the first spin-up solve. Do not run until upstream stages and cluster config are ready.
#
# md = org.loadmodel("Parameterization")
# md = setflowequation(md, "SSA", "all")
#
# if friction_style == "coulomb":
#     ...  # apply Coulomb override from the previous section
#
# md.timestepping.time_step = 1
# md.timestepping.final_time = 200000
# md.settings.output_frequency = 2000
# md.settings.checkpoint_frequency = 2000
# md.stressbalance.maxiter = 30
# md.cluster = cluster
# md.miscellaneous.name = "MISMIP_" + modelname + "_Tss1"
#
# md = solve(md, "tr", "loadonly", loadonly, "lock", lock, "runtimename", False)
# org.savemodel(md)

## 13. Restart stages

The `Transient_steadystate2`, `Transient_steadystate3`, `Transient_steadystate4`, and `Transient_steadystate5` stages all follow the same restart logic.

Each one loads the previous stage, copies the last transient result back into model fields, and runs another segment.

The copied fields are usually:

- `Vx`, `Vy`, `Vel` into `md.initialization`
- `Thickness`, `Base`, `Surface` into `md.geometry`
- `MaskOceanLevelset` into `md.mask.ocean_levelset`

This makes the next solve start from the previous final state rather than from the original analytic initialization.

In [ ]:
# Reusable pattern behind the restart stages.
#
# previous = md.results.TransientSolution[-1]
# md.initialization.vx = previous.Vx
# md.initialization.vy = previous.Vy
# md.initialization.vel = previous.Vel
# md.geometry.thickness = previous.Thickness
# md.geometry.base = previous.Base
# md.geometry.surface = previous.Surface
# md.mask.ocean_levelset = previous.MaskOceanLevelset

## 14. Extrusion stage

`Transient_extrude` creates a 3D model from a spun-up 2D model:

```python
md = md.extrude(10, 1.1)
md = setflowequation(md, 'HO', 'all')
```

This creates 10 vertical layers, with layer spacing controlled by the exponent-like second argument `1.1`.

After extrusion, the script disables thermal and SMB transients and initializes temperature to `273 K`.

## 15. Solver branch choices

Once the spin-up state exists, `runme.py` branches into solver comparisons.

| Branch | Flow equation | Mesh handling | Main purpose |
| --- | --- | --- | --- |
| `GlenSSA` | `SSA` | collapse to 2D | Shallow-shelf baseline comparison. |
| `GlenMOLHO` | `MOLHO` | collapse to 2D | Modified/layered higher-order style comparison. |
| `GlenHO` | `HO` | uses extruded model | Higher-order comparison. |
| `GlenFS` | `FS` | extrudes to 3D | Short full-Stokes-style experiment. |

All branches request diagnostic outputs such as volume, grounded area, strain rates, and masks. The exact requested outputs depend on the flow equation.

## 16. Example: current target branch is likely `GlenMOLHO`

Because `steps = [11]`, the current `runme.py` appears to target the `GlenMOLHO` block if the organizer counts stages in source order.

The logical path for that target is:

1. `Mesh_generation`
2. `Parameterization`
3. `Transient_Steadystate`
4. optional restart/spin-up stages
5. `Transient_extrude` unless using model numbers 5 or 6
6. `GlenMOLHO`

The `GlenMOLHO` branch then collapses the model, sets the flow equation to `MOLHO`, adds a stress-balance toolkit option, applies `SetMOLHOBC(md)`, and runs a 1000-year transient with monthly time steps.

In [ ]:
# Skeleton of the GlenMOLHO branch from runme.py.
#
# if modelnum in (5, 6):
#     md = org.loadmodel("Transient_Steadystate")
# else:
#     md = org.loadmodel("Transient_extrude")
#
# md.transient.requested_outputs = [
#     "default", "IceVolume", "IceVolumeAboveFloatation", "GroundedArea",
#     "VxShear", "VyShear", "VxBase", "VyBase", "VxSurface", "VySurface",
#     "VxAverage", "VyAverage", "StrainRatexx", "StrainRatexy", "StrainRateyy",
#     "StrainRateeffective", "MaskOceanLevelset", "IceMaskNodeActivation", "MaskIceLevelset",
# ]
#
# md = md.collapse()
# md = setflowequation(md, "MOLHO", "all")
# md.toolkits = toolkits.addoptions(md.toolkits, "StressbalanceAnalysis", bcgslbjacobioptions())
# md = SetMOLHOBC(md)
#
# md.timestepping.time_step = 1.0 / 12.0
# md.timestepping.final_time = 1000
# md.settings.output_frequency = 600
# md.cluster = cluster
# md.miscellaneous.name = "MISMIP_" + modelname + "_GMOLHO"
#
# md = solve(md, "tr")
# org.savemodel(md)

## 17. Enhanced and ESTAR branches

The later branches reuse the same prepared spin-up state but alter the material model.

Enhanced Glen branches use `matenhancedice(...)` and set an enhancement factor like:

```python
md.materials.rheology_E = 5.0 * np.ones(md.mesh.numberofvertices)
```

ESTAR branches use `matestar(...)` and set `rheology_Es` and `rheology_Ec`.

These branches look experimental in the current file. Several lines appear to contain typos or unresolved variables, so they should be reviewed before running.

## 18. Practical configuration checklist

Before running the model, check these choices in order:

- Pick `modelnum`: this decides resolution and friction family.
- Decide `steps`: start with upstream stages if saved models do not already exist.
- Check `ISSM_DIR`: imports depend on it.
- Check cluster paths: the current Gadi paths are user-specific.
- Confirm `Domain.exp` and `Front.exp`: `Domain.exp` is present, while `Mismip.py` references `Front.exp`.
- Run `Mesh_generation` before `Parameterization`.
- Run `Parameterization` before any solve.
- Run a spin-up before solver-comparison branches.
- Treat later enhanced/ESTAR branches as work-in-progress until syntax and missing variables are fixed.

For learning, the safest first path is usually: `Mesh_generation`, `Parameterization`, then inspect `md` before launching long transient solves.

## 19. Summary

The clean logical path is:

```text
model choice -> mesh -> physical parameterization -> spin-up -> solver branch -> comparison/analysis
```

`runme.py` gives the pipeline and branch structure. `Mismip.py` gives the physical configuration. Reading them together makes the model much easier to understand than reading either file alone.